# Olist: chọn thời điểm dự đoán, rồi đổi metric ra tiền

Bài toán: cảnh báo đơn có nguy cơ giao trễ để CSKH nhắn tin xin lỗi và báo ngày giao mới **trước khi**
đơn tới tay khách. Vì can thiệp trước khi giao, mọi thông tin sau thời điểm dự đoán (ngày giao thực tế,
review) đều là leak. Đây là điểm khác với model review xấu "lúc đơn vừa giao" của module 10/11.

Hai thời điểm ứng viên:
- **purchase**: khách vừa đặt hàng.
- **carrier**: seller vừa giao đơn cho đơn vị vận chuyển (biết thêm số ngày seller xử lý, số ngày còn lại tới ngày hẹn).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, gọi hàm, vẽ. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/15-ml-system-design/exercises/intermediate/01-olist-decision-time -q`

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
orders.shape, items.shape

## 1. Bảng một dòng mỗi đơn

Mỗi feature phải trả lời được câu "lúc dự đoán đã có giá trị này chưa?". `handling_days` và
`remaining_days` chỉ có từ lúc giao cho vận chuyển.

In [ ]:
def build_orders(orders: pd.DataFrame, items: pd.DataFrame, customers: pd.DataFrame,
                 sellers: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong (delivered + có order_delivered_customer_date).

    Cột: order_id, purchase_ts, late (int 0/1), promised_days, same_state (int, 1 nếu MỌI seller của đơn
    cùng bang với khách), log_freight (log1p tổng freight_value), n_items, n_sellers, customer_state,
    handling_days (giao cho vận chuyển − mua), remaining_days (ngày hẹn − giao cho vận chuyển).
    Mọi khoảng thời gian tính bằng ngày, số thực. Đơn thiếu ngày giao vận chuyển để NaN.
    """
    o = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    o = o.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
    it = items.merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
    it = it.merge(o[["order_id", "customer_state"]], on="order_id", how="inner")
    it["same"] = it.seller_state == it.customer_state
    agg = it.groupby("order_id").agg(n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
                                     freight=("freight_value", "sum"), same_state=("same", "all"))
    o = o.merge(agg, on="order_id", how="left")
    day = pd.Timedelta(days=1)
    return pd.DataFrame({
        "order_id": o.order_id,
        "purchase_ts": o.order_purchase_timestamp,
        "late": (o.order_delivered_customer_date > o.order_estimated_delivery_date).astype(int),
        "promised_days": (o.order_estimated_delivery_date - o.order_purchase_timestamp) / day,
        "same_state": o.same_state.fillna(False).astype(int),
        "log_freight": np.log1p(o.freight.fillna(0.0)),
        "n_items": o.n_items.fillna(0).astype(int),
        "n_sellers": o.n_sellers.fillna(0).astype(int),
        "customer_state": o.customer_state,
        "handling_days": (o.order_delivered_carrier_date - o.order_purchase_timestamp) / day,
        "remaining_days": (o.order_estimated_delivery_date - o.order_delivered_carrier_date) / day,
    }).reset_index(drop=True)


In [ ]:
# explore
df = build_orders(orders, items, customers, sellers)
print(len(df), df.late.mean())
df.describe()

## 2. Chia theo thời gian (định nghĩa chung của track)

In [ ]:
def time_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """(train, val, test) theo purchase_ts: < 2018-01-01 | [2018-01-01, 2018-04-01) | [2018-04-01, 2018-09-01)."""
    t = df.purchase_ts
    return (df[t < "2018-01-01"], df[(t >= "2018-01-01") & (t < "2018-04-01")],
            df[(t >= "2018-04-01") & (t < "2018-09-01")])


In [ ]:
# explore: prevalence khác nhau giữa ba tập (khủng hoảng 02–03/2018 nằm trong validation)
train, val, test = time_split(df)
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, len(d), round(d.late.mean(), 4))

## 3. Feature hợp lệ theo thời điểm quyết định

In [ ]:
def features_at(decision_time: str) -> list[str]:
    """Các cột feature hợp lệ tại thời điểm quyết định: "purchase" (khách vừa đặt) hoặc "carrier" (đơn vừa
    giao cho vận chuyển). Thời điểm khác thì ValueError.
    """
    base = ["promised_days", "same_state", "log_freight", "n_items", "n_sellers", "customer_state"]
    if decision_time == "purchase":
        return base
    if decision_time == "carrier":
        return base + ["handling_days", "remaining_days"]
    raise ValueError(f"Không biết thời điểm {decision_time!r}")


## 4. Metric offline và giá trị tiền

In [ ]:
def offline_report(y, p, ks=(100, 500, 1000)) -> dict[str, float]:
    """prevalence, roc_auc, pr_auc (average precision) và precision@k cho mỗi k (khoá "precision@<k>")."""
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    order = np.argsort(-p, kind="stable")
    out = {"prevalence": float(y.mean()), "roc_auc": float(roc_auc_score(y, p)),
           "pr_auc": float(average_precision_score(y, p))}
    for k in ks:
        out[f"precision@{k}"] = float(y[order[:k]].mean())
    return out


In [ ]:
def value_table(y, p, fractions, cost_action: float, loss_event: float, effect: float,
                n_weeks: int) -> pd.DataFrame:
    """Mỗi tỉ lệ f trong fractions: chọn top k = floor(f * n) theo p. Index fraction; cột k, precision,
    value = TP * effect * loss_event − k * cost_action (R$ trên cả tập), value_per_week = value / n_weeks.
    """
    y = np.asarray(y)
    order = np.argsort(-np.asarray(p, dtype=float), kind="stable")
    rows = []
    for f in fractions:
        k = int(np.floor(f * len(y)))
        tp = int(y[order[:k]].sum())
        value = tp * effect * loss_event - k * cost_action
        rows.append({"fraction": f, "k": k, "precision": tp / k if k else float("nan"),
                     "value": value, "value_per_week": value / n_weeks})
    return pd.DataFrame(rows).set_index("fraction")


## 5. Model

In [ ]:
def fit_predict(train: pd.DataFrame, test: pd.DataFrame, features: list[str]) -> np.ndarray:
    """Train một classifier dự đoán `late` trên train[features], trả xác suất cho test (mảng 1 chiều).
    Phải xử lý được NaN (handling_days) và cột chữ customer_state. Model nào cũng được.
    """
    states = pd.CategoricalDtype(sorted(train["customer_state"].dropna().unique())) if "customer_state" in features else None

    def matrix(d):
        x = d[features].copy()
        if states is not None:
            x["customer_state"] = x["customer_state"].astype(states)
        return x

    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, min_child_samples=50,
                               verbose=-1, random_state=0)
    model.fit(matrix(train), train["late"])
    return model.predict_proba(matrix(test))[:, 1]


In [ ]:
# explore: so hai thời điểm trên test (train trên train)
preds = {}
for t in ["purchase", "carrier"]:
    preds[t] = fit_predict(train, test, features_at(t))
    print(t, {k: round(v, 3) for k, v in offline_report(test.late, preds[t]).items()})

In [ ]:
# explore: đổi ra tiền. Giả định: tin nhắn 0.5 R$, review xấu thiệt 30 R$, nhắn tin giảm 15 điểm % xác suất review xấu
weeks = test.purchase_ts.dt.to_period("W").nunique()
for t, p in preds.items():
    print(t)
    display(value_table(test.late, p, [0.01, 0.02, 0.05, 0.10, 0.20], 0.5, 30.0, 0.15, weeks).round(3))

In [ ]:
# explore: baseline không ML. Luật "còn dưới r ngày tới ngày hẹn lúc giao vận chuyển"
for r in [0, 3, 5, 7]:
    rule = (test.remaining_days < r).to_numpy()
    tp = test.late[rule].sum()
    print(r, rule.sum(), round(tp / rule.sum(), 3), round(tp / test.late.sum(), 3),
          round((tp * 0.15 * 30 - rule.sum() * 0.5) / weeks, 1), "R$/tuần")

## 6. Câu hỏi

1. PR-AUC của model lúc đặt hàng trên validation và test khác nhau bao nhiêu? Bao nhiêu phần do model,
   bao nhiêu do prevalence? (gợi ý: chia PR-AUC cho prevalence).
2. Đợi tới lúc giao vận chuyển đổi được gì, mất gì? Còn bao nhiêu ngày để hành động?
3. Vì sao precision@100 của model lúc giao vận chuyển gần 1? Đó có phải "model giỏi" không?
4. Luật một dòng so với model thế nào về giá trị mỗi tuần? Bạn sẽ ra mắt cái nào trước, vì sao?
5. Giá trị mỗi tuần có đáng xây hệ thống không? Giả định nào bạn cần business xác nhận?

## Trả lời (lời giải)

1. Validation 0.274 (prevalence 14.6%), test 0.120 (6.0%): lift 1.9 và 2.0 lần. Model không tệ đi;
   prevalence giảm. Luôn so PR-AUC với prevalence.
2. Thêm khoảng 2.2 ngày (trung vị thời gian seller xử lý), PR-AUC từ 0.120 lên 0.378; vẫn còn trung vị
   20.3 ngày tới ngày hẹn để hành động.
3. 226 đơn trên test được giao vận chuyển **sau** ngày hẹn (`remaining_days < 0`): chắc chắn trễ. Model
   chỉ học lại một luật hiển nhiên; giá trị thật nằm ở phần còn lại của thứ hạng.
4. Luật "còn dưới 5 ngày" cho khoảng 99 R$/tuần, ngang model top 10% (khoảng 96 R$/tuần). Ra mắt luật
   trước: rẻ, giải thích được, tạo log quyết định và holdout; model thay vào khi chứng minh hơn luật.
5. Khoảng 100 R$/tuần là quá nhỏ cho một hệ thống ML có người bảo trì. Giả định cần xác nhận: thiệt hại
   một review xấu (30 R$) và tác động của tin nhắn (15 điểm %). Xem bài Advanced 02.